# Proyecto de clasificación multiclase: notebook general

Sigue la *Guía de un proyecto de clasificación en ML* (pasos 0 a 10). El análisis exploratorio previo vive aparte en `01_eda.ipynb`.

## 0. Definir el problema

> Intro del EDA (pista Machine Learning).

- **Contexto:** conjunto *Eucalyptus Soil Conservation* (Bruce Bulloch, colección WEKA, OpenML id=188): 736 filas y 19 variables de ensayos de semillas de eucalipto en la Isla Norte de Nueva Zelanda. El objetivo original era determinar qué lotes de semillas son mejores para la conservación del suelo en colinas de clima estacionalmente seco.
- **Tarea:** clasificación multiclase, 5 clases.
- **Variable objetivo:** `Utility` (utilidad del lote): `good` 214 (29,1 %), `none` 180 (24,5 %), `average` 130 (17,7 %), `low` 107 (14,5 %), `best` 105 (14,3 %). Es una escala ordenada: `none` < `low` < `average` < `good` < `best`.
- **Métrica de éxito:** F1 macro (trata las 5 clases por igual) más accuracy global; además precision, recall y F1 por clase.
- **Baseline a superar:** `DummyClassifier` (clase mayoritaria, ≈ 29,1 % de accuracy) y regresión logística simple.
- **Error más costoso:** ninguna clase en particular; se tratan todas las confusiones por igual, pero en el análisis de errores se marcan las más alejadas en la escala (p. ej. `best` predicho como `none`).
- **Restricción del proyecto:** sobreajuste < 5 % entre entrenamiento y validación.
- **Límites de los datos:** los métodos de ensayo cambiaron con el tiempo, lo que puede reducir la comparabilidad entre sitios; faltan datos de supervivencia, vigor, resistencia a insectos y forma (tallo, copa, ramas), sobre todo en la estación de Morea, y no se sabe si es por muerte de las plantas o por falta de medición.
- **Pendiente de decidir:** qué decisión concreta se tomará con la predicción y si todas las variables (p. ej. supervivencia, vigor) estarán disponibles en el momento de predecir (riesgo de *data leakage*, se revisa en la exploración).

## 1. Datos y split

In [ ]:
# Librería común del equipo (comun.py, ver docs/acuerdos.md)
from comun import SEMILLA, cargar_openml, dividir  # noqa: F401

In [ ]:
# Eucalyptus (OpenML id=188); se guarda en caché local tras la primera descarga
X, y = cargar_openml()

print(f"{X.shape[0]} filas, {X.shape[1]} variables")
print(y.value_counts())

In [ ]:
# Split estratificado 70/15/15. El test no se toca hasta la evaluación final (paso 8).
X_train, X_val, X_test, y_train, y_val, y_test = dividir(X, y)

print(f"train: {len(X_train)} | validación: {len(X_val)} | test: {len(X_test)}")

## 2. EDA (resumen)

El análisis exploratorio completo está en [`01_eda.ipynb`](01_eda.ipynb). Aquí solo se recogen sus conclusiones y las variables.

### Qué ha encontrado el EDA (conjunto de entrenamiento, n = 515)
- Sin duplicados; un error evidente en `DBH` (42085); ausentes en dos bloques (`Mor`/`Wen` sin ninguna medida, `Cly` sin `Surv`) y 5 en `PMCno`.
- **Las 46 filas sin medidas son todas `none`** (46 de 46): la ausencia delata la clase, hay riesgo de *data leakage*.
- `Vig` es la variable más ligada a la clase; el sitio condiciona mucho la clase y siete columnas describen lo mismo (el sitio).
- **Decisión del equipo (ADR-0001):** las parcelas sin medidas se quitan solo del entrenamiento; validación y test las conservan y se predicen `none` por regla fija, con las métricas informadas con y sin ellas.
- **Dónde acaba el EDA:** aplica las decisiones del equipo. La limpieza común (`docs/acuerdos.md`) y la exportación a `data/` están en la sección 10 de `01_eda.ipynb`; imputar ausentes, codificar, escalar y el resto del preprocesado lo hace cada integrante en su `Pipeline` (lista en la conclusión de `01_eda.ipynb`).

### Variables
| Variable | Descripción | Tipo | Lo que dice el EDA |
|---|---|---|---|
| `Abbrev` | Abreviatura del sitio | categórica (16) | El sitio condiciona la clase; `Mor` (41) y `Wen` (5) son todo `none` |
| `Rep` | Réplica en el sitio | entera | Muy asimétrica (máx. 22, mediana 2) |
| `Locality`, `Map_Ref`, `Latitude` | Localidad, referencia de mapa y latitud | categóricas | Un único valor por sitio: redundantes con `Abbrev` |
| `Altitude`, `Rainfall`, `Frosts`, `Year` | Altitud, lluvia (mm/año), heladas (°C) y año de plantación | numéricas | Un único valor por sitio; `Altitude` casi categórica |
| `Sp` | Código de especie | categórica (27) | 4 especies con menos de 5 filas |
| `PMCno` | Número de lote de semillas | entera | 5 ausentes; ligada a la clase muy débilmente (ρ = -0,04); parece un identificador |
| `DBH` | Diámetro a la altura del pecho (cm) | real | Error de 42085 en una fila; ρ = 0,34 con la clase; 0,82 con `Ht` |
| `Ht` | Altura (m) | real | ρ = 0,45 con la clase |
| `Surv` | Supervivencia | entera | 68 ausentes (46 en `Mor`/`Wen`, 22 en `Cly`); ρ = 0,27 |
| `Vig` | Vigor | real | 46 ausentes (`Mor`/`Wen`); la más ligada a la clase (ρ = 0,79) |
| `Ins_res` | Resistencia a insectos | real | 46 ausentes; ρ = 0,46 |
| `Stem_Fm`, `Crown_Fm`, `Brnch_Fm` | Forma del tallo, copa y ramas | reales | 46 ausentes cada una; correlacionadas entre sí (0,56 a 0,70); ρ = 0,46, 0,40 y 0,12 |
| `Utility` | **Objetivo**: utilidad del lote | categórica ordenada (5) | `good` 29,1 %, `none` 24,5 %, `average` 17,7 %, `low` 14,6 %, `best` 14,2 % |

ρ = correlación de Spearman con la clase ordenada, en el conjunto de entrenamiento (por pares con dato).

## 3. Preprocesado (`Pipeline` + `ColumnTransformer`, ajustar solo con train)

## 4. Feature engineering y selección

## 5. Baseline (`DummyClassifier` y regresión logística)

## 6. Entrenamiento y validación (`StratifiedKFold`, 5 folds)

## 7. Ajuste de hiperparámetros

## 8. Evaluación final (una sola vez sobre test)

## 9. Análisis de errores e interpretación

## 10. Despliegue y monitoreo